## Using `soccerdata` to scrape FBref data

In [36]:
import soccerdata as sd
print(sd.FBref.available_leagues())

['ARG-Primera Division', 'AUT-Bundesliga', 'BEL-First Division A', 'BRA-Serie A', 'Big 5 European Leagues Combined', 'CZE-First League', 'ENG-Premier League', 'ESP-La Liga', 'FRA-Ligue 1', 'GER-Bundesliga', 'GRE-Super League', 'INT-European Championship', "INT-Women's World Cup", 'INT-World Cup', 'ITA-Serie A', 'JPN-J1 League', 'KOR-K League 1', 'MEX-Liga MX', 'NED-Eredivisie', 'POR-Primeira Liga', 'SAU-Saudi Pro League', 'SCO-Premiership', 'SUI-Super League', 'TUR-Super Lig', 'USA-MLS']


In [37]:
import soccerdata as sd
import pandas as pd

leagues = [
    "ENG-Premier League", "ESP-La Liga", "FRA-Ligue 1", "GER-Bundesliga", "ITA-Serie A",
    "NED-Eredivisie", "POR-Primeira Liga", "BRA-Serie A", "ARG-Primera Division",
    "USA-MLS", "MEX-Liga MX", "SAU-Saudi Pro League", "TUR-Super Lig",
    "BEL-First Division A", "SCO-Premiership", "GRE-Super League", "CZE-First League",
    "AUT-Bundesliga", "SUI-Super League", "KOR-K League 1", "JPN-J1 League", "ENG-Championship"
]

fbref = sd.FBref(leagues=leagues, seasons=2025)
stats = fbref.read_player_season_stats(stat_type="standard")
print(f"Total players: {stats.shape[0]}")

# Re-run coverage check
players = pd.read_csv("../data/processed/player_fixtures.csv")
wc_players = players[["player", "team", "position"]].drop_duplicates()

stats_names = set(stats.index.get_level_values("player").str.lower().str.strip())
wc_players["name_lower"] = wc_players["player"].str.lower().str.strip()

matched = wc_players[wc_players["name_lower"].isin(stats_names)]
unmatched = wc_players[~wc_players["name_lower"].isin(stats_names)]

print(f"Matched: {len(matched)} ({len(matched)/len(wc_players)*100:.1f}%)")
print(f"Unmatched: {len(unmatched)} ({len(unmatched)/len(wc_players)*100:.1f}%)")

match_rate_by_team = (
    wc_players.groupby("team")
    .apply(lambda g: g["name_lower"].isin(stats_names).mean())
    .sort_values()
)
print("\nTeams with lowest match rates:")
print(match_rate_by_team.head(20))
print("\nTeams with highest match rates:")
print(match_rate_by_team.tail(10))

ValueError: 
                        Invalid league 'ENG-Championship'. Valid leagues are:
                        ['ARG-Primera Division',
 'AUT-Bundesliga',
 'BEL-First Division A',
 'BRA-Serie A',
 'Big 5 European Leagues Combined',
 'CZE-First League',
 'ENG-Premier League',
 'ESP-La Liga',
 'FRA-Ligue 1',
 'GER-Bundesliga',
 'GRE-Super League',
 'INT-European Championship',
 "INT-Women's World Cup",
 'INT-World Cup',
 'ITA-Serie A',
 'JPN-J1 League',
 'KOR-K League 1',
 'MEX-Liga MX',
 'NED-Eredivisie',
 'POR-Primeira Liga',
 'SAU-Saudi Pro League',
 'SCO-Premiership',
 'SUI-Super League',
 'TUR-Super Lig',
 'USA-MLS']
                        

## Data Merging

In [25]:
import unicodedata
from rapidfuzz import process, fuzz

def to_ascii(name):
    return unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode().lower().strip()

# --- Flatten FBref MultiIndex columns ---
stats_flat = stats.copy()
stats_flat.columns = [
    f"{b}" if not a or a == b else f"{a}_{b}"
    for a, b in stats_flat.columns
]
stats_reset = stats_flat.reset_index()
stats_reset["name_ascii"] = stats_reset["player"].apply(to_ascii)

# --- WC players ---
wc_players = pd.read_csv("../data/processed/player_fixtures.csv")[
    ["player", "team", "position"]
].drop_duplicates()
wc_players["name_ascii"] = wc_players["player"].apply(to_ascii)

# --- Pass 1: exact ASCII match ---
fbref_ascii_set = set(stats_reset["name_ascii"])
exact_matched = wc_players[wc_players["name_ascii"].isin(fbref_ascii_set)].copy()
unmatched = wc_players[~wc_players["name_ascii"].isin(fbref_ascii_set)].copy()
print(f"Exact matches: {len(exact_matched)} | Unmatched: {len(unmatched)}")

# --- Pass 2: fuzzy match constrained by nationality ---
TEAM_TO_NATION = {
    "Algeria": "ALG", "Argentina": "ARG", "Australia": "AUS",
    "Austria": "AUT", "Belgium": "BEL", "Bosnia and Herzegovina": "BIH",
    "Brazil": "BRA", "Cabo Verde": "CPV", "Canada": "CAN", "Chile": "CHI",
    "Colombia": "COL", "Congo DR": "COD", "Costa Rica": "CRC",
    "Croatia": "CRO", "Curaçao": "CUW", "Côte d'Ivoire": "CIV",
    "Czechia": "CZE", "Ecuador": "ECU", "Egypt": "EGY", "England": "ENG",
    "France": "FRA", "Germany": "GER", "Ghana": "GHA", "Greece": "GRE",
    "Haiti": "HAI", "Honduras": "HON", "Hungary": "HUN", "IR Iran": "IRN",
    "Iraq": "IRQ", "Japan": "JPN", "Jordan": "JOR", "Kenya": "KEN",
    "Korea Republic": "KOR", "Mexico": "MEX", "Morocco": "MAR",
    "Netherlands": "NED", "New Zealand": "NZL", "Nigeria": "NGA",
    "Norway": "NOR", "Panama": "PAN", "Paraguay": "PAR", "Peru": "PER",
    "Poland": "POL", "Portugal": "POR", "Qatar": "QAT", "Romania": "ROU",
    "Saudi Arabia": "KSA", "Scotland": "SCO", "Senegal": "SEN",
    "Serbia": "SRB", "Slovakia": "SVK", "Slovenia": "SVN",
    "South Africa": "RSA", "Spain": "ESP", "Sweden": "SWE",
    "Switzerland": "SUI", "Trinidad and Tobago": "TRI", "Tunisia": "TUN",
    "Türkiye": "TUR", "Ukraine": "UKR", "Uruguay": "URU", "USA": "USA",
    "Uzbekistan": "UZB", "Venezuela": "VEN",
}

THRESHOLD = 85
KNOWN_FALSE_POSITIVES = {("Weverton", "Brazil")}

fbref_by_nation = stats_reset[["player", "name_ascii", "nation_"]].drop_duplicates()

fuzzy_rows = []
for _, row in unmatched.iterrows():
    nation_code = TEAM_TO_NATION.get(row["team"])
    if not nation_code:
        continue
    candidates = fbref_by_nation[fbref_by_nation["nation_"] == nation_code]
    if candidates.empty:
        continue
    result = process.extractOne(
        row["name_ascii"],
        candidates["name_ascii"].tolist(),
        scorer=fuzz.token_sort_ratio,
        score_cutoff=THRESHOLD,
    )
    if result and (row["player"], row["team"]) not in KNOWN_FALSE_POSITIVES:
        matched_ascii, score, _ = result
        fuzzy_rows.append({
            "player": row["player"],
            "team": row["team"],
            "position": row["position"],
            "name_ascii": matched_ascii,  # use FBref's ascii so the downstream merge works
        })

fuzzy_matched = pd.DataFrame(fuzzy_rows) if fuzzy_rows else pd.DataFrame(columns=exact_matched.columns)
print(f"Fuzzy matches: {len(fuzzy_matched)}")

matched_players = pd.concat([exact_matched, fuzzy_matched], ignore_index=True)
print(f"Total matched: {len(matched_players)} out of {len(wc_players)} WC players")


Exact matches: 784 | Unmatched: 626
Fuzzy matches: 46
Total matched: 830 out of 1410 WC players


In [28]:
matched_players.head(10)

,player,team,position,name_ascii
0,Rayan Aït-Nouri,Algeria,DEF,rayan ait-nouri
1,Ramy Bensebaini,Algeria,DEF,ramy bensebaini
2,Aïssa Mandi,Algeria,DEF,aissa mandi
3,Rafik Belghali,Algeria,DEF,rafik belghali
4,Amine Gouiri,Algeria,FWD,amine gouiri
5,Riyad Mahrez,Algeria,MID,riyad mahrez
6,Ibrahim Maza,Algeria,FWD,ibrahim maza
7,Anis Hadj Moussa,Algeria,FWD,anis hadj moussa
8,Ramiz Zerrouki,Algeria,MID,ramiz zerrouki
9,Hicham Boudaoui,Algeria,MID,hicham boudaoui


Now, we just want to explore the 580 unmatched players 

In [29]:
unmatched_final = wc_players[
    ~wc_players["player"].isin(matched_players["player"])
][["player", "team", "position"]].sort_values(["team", "position"]).reset_index(drop=True)

print(f"Unmatched players: {len(unmatched_final)}")
print(unmatched_final.to_string())

Unmatched players: 580
                          player                    team position
0                   Mehdi Dorval                 Algeria      DEF
1               Zinéddine Belaïd                 Algeria      DEF
2                    Sohaib Nair                 Algeria      DEF
3                   Achref Abada                 Algeria      DEF
4                Farès Ghedjemis                 Algeria      FWD
5                Ahmed Benbouali                 Algeria      FWD
6                   Amin Chiakha                 Algeria      FWD
7                Anthony Mandréa                 Algeria       GK
8                    Luca Zidane                 Algeria       GK
9                  Melvin Mastil                 Algeria       GK
10              Kilian Belazzoug                 Algeria       GK
11                 Adil Boulbina                 Algeria      MID
12               Yacine Titraoui                 Algeria      MID
13                  Marcos Acuña               Argent

Manual Mapping

In [33]:
# Pass 3: manual overrides
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from data.manual_overrides import MANUAL_OVERRIDES


still_unmatched = wc_players[~wc_players["player"].isin(matched_players["player"])].copy()

override_rows = []
for _, row in still_unmatched.iterrows():
    fbref_name = MANUAL_OVERRIDES.get(row["player"])
    if fbref_name:
        fbref_ascii = to_ascii(fbref_name)
        override_rows.append({
            "player": row["player"],
            "team": row["team"],
            "position": row["position"],
            "name_ascii": fbref_ascii,
        })

override_matched = pd.DataFrame(override_rows) if override_rows else pd.DataFrame(columns=matched_players.columns)
print(f"Manual override matches: {len(override_matched)}")

matched_players = pd.concat([matched_players, override_matched], ignore_index=True)
print(f"Total matched: {len(matched_players)} / {len(wc_players)} WC players")

Manual override matches: 33
Total matched: 863 / 1410 WC players


In [34]:
unmatched_final = wc_players[
    ~wc_players["player"].isin(matched_players["player"])
][["player", "team", "position"]].sort_values(["team", "position"]).reset_index(drop=True)

print(f"Unmatched players: {len(unmatched_final)}")
print(unmatched_final.to_string())

Unmatched players: 547
                        player                    team position
0                 Mehdi Dorval                 Algeria      DEF
1             Zinéddine Belaïd                 Algeria      DEF
2                  Sohaib Nair                 Algeria      DEF
3                 Achref Abada                 Algeria      DEF
4              Farès Ghedjemis                 Algeria      FWD
5              Ahmed Benbouali                 Algeria      FWD
6                 Amin Chiakha                 Algeria      FWD
7              Anthony Mandréa                 Algeria       GK
8                  Luca Zidane                 Algeria       GK
9                Melvin Mastil                 Algeria       GK
10            Kilian Belazzoug                 Algeria       GK
11               Adil Boulbina                 Algeria      MID
12             Yacine Titraoui                 Algeria      MID
13                Marcos Acuña               Argentina      DEF
14       Lucas Ma

Here, we merge the `matched_players` dataframe with the relevant FBRef data

In [38]:
import numpy as np

# Step 1: Merge with league included
merged = matched_players.merge(
    stats_reset[[
        "name_ascii", "league", "pos_",
        "Playing Time_Min",
        "Per 90 Minutes_Gls", "Per 90 Minutes_Ast", "Per 90 Minutes_G+A"
    ]],
    on="name_ascii",
    how="left"
).rename(columns={
    "pos_": "fbref_pos",
    "Playing Time_Min": "minutes",
    "Per 90 Minutes_Gls": "gls_p90",
    "Per 90 Minutes_Ast": "ast_p90",
    "Per 90 Minutes_G+A": "ga_p90"
})

# Step 2: Apply minimum minutes filter
MIN_MINS = 900
merged_filtered = merged[merged["minutes"] >= MIN_MINS].copy()
print(f"Players after {MIN_MINS} min filter: {len(merged_filtered)}")

# Step 3: League-normalise within position group
# Use the full stats_reset (not just WC players) as the reference population
# so percentiles are relative to all players in that league, not just WC squad players
ref = stats_reset[stats_reset["Playing Time_Min"] >= MIN_MINS].copy().rename(columns={
    "pos_": "fbref_pos",
    "Playing Time_Min": "minutes",
    "Per 90 Minutes_Gls": "gls_p90",
    "Per 90 Minutes_Ast": "ast_p90",
    "Per 90 Minutes_G+A": "ga_p90"
})

for stat in ["gls_p90", "ast_p90", "ga_p90"]:
    ref[f"{stat}_pct"] = ref.groupby(["league", "fbref_pos"])[stat].rank(pct=True)

# Merge percentiles back via a join instead of index mapping
pct_cols = ["name_ascii", "league", "gls_p90_pct", "ast_p90_pct", "ga_p90_pct"]
merged_filtered = merged_filtered.merge(
    ref[pct_cols].drop_duplicates(subset=["name_ascii", "league"]),
    on=["name_ascii", "league"],
    how="left"
)

Players after 900 min filter: 765


## Weight Table

In [39]:
# Build weight table for matched non-GK players
matched_weights = merged_filtered[merged_filtered["fbref_pos"] != "GK"][[
    "player", "team", "position", "gls_p90_pct", "ast_p90_pct"
]].copy()
matched_weights["weight_source"] = "fbref"

# # Unmatched players - use price rank as proxy for both percentiles
# all_players = pd.read_csv("../data/processed/player_fixtures.csv")[
#     ["player", "team", "position", "price"]
# ].drop_duplicates()

# unmatched_players = all_players[
#     ~all_players["player"].isin(matched_weights["player"])
# ].copy()
# unmatched_players["gls_p90_pct"] = (
#     unmatched_players.groupby(["team", "position"])["price"].rank(pct=True)
# )
# unmatched_players["ast_p90_pct"] = unmatched_players["gls_p90_pct"]
# unmatched_players["weight_source"] = "price_fallback"
# unmatched_players = unmatched_players.drop(columns=["price"])

# Combine and dedup (keep highest minutes for fbref duplicates)
# weight_table = pd.concat([matched_weights, unmatched_players], ignore_index=True)
weight_table = matched_weights.copy()

mins_map = merged_filtered.set_index(["player", "team"])["minutes"].to_dict()
weight_table["minutes"] = weight_table.apply(
    lambda r: mins_map.get((r["player"], r["team"]), 0), axis=1
)
weight_table = (
    weight_table
    .sort_values("minutes", ascending=False)
    .drop_duplicates(subset=["player", "team"], keep="first")
    .drop(columns=["minutes"])
    .reset_index(drop=True)
)

# Override all GKs with neutral weights
weight_table.loc[weight_table["position"] == "GK", "gls_p90_pct"] = 0.5
weight_table.loc[weight_table["position"] == "GK", "ast_p90_pct"] = 0.5
weight_table.loc[weight_table["position"] == "GK", "weight_source"] = "gk_neutral"

weight_table.to_csv("../data/weight_table.csv", index=False)